# Step 05 Decomposition Demo

Runs the v3.2 algorithm (via `proto3.geometry.decompose.run`, the proto3 mm-friendly entry) on all 6 fixtures (A1, A2, B1, R1, R2, D1) and visualizes pieces + atoms colored by family.

## Prerequisites

1. `python -m pip install -e ".[dev]"` (one-time, from repo root)
2. `nbstripout --install` (one-time)
3. Repo root resolved automatically by walking up.

## Unit conversion (R-S05-7 mitigated)

proto3 schema uses **mm** (D006); v3.2 algorithm uses **m**. Step 05 review-followup introduced `proto3.geometry.decompose.run()` — a mm-friendly wrapper that handles the conversion internally. This notebook uses `run()` directly with mm coordinates from `BuildingInput.floors[*].footprint`; the previous inline `(x / 1000)` conversion is no longer needed.

## Output convention (S03-D13)

Generated PNGs go to `outputs/notebooks/step05_decomposition/<run_id>/`.

In [ ]:
from pathlib import Path


def _find_repo_root(start: Path) -> Path:
    p = start.resolve()
    for candidate in [p, *p.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError(f"pyproject.toml not found from {start} upward")


ROOT = _find_repo_root(Path.cwd())
print("repo root:", ROOT)

In [ ]:
import sys

if str(ROOT / "tests") not in sys.path:
    sys.path.insert(0, str(ROOT / "tests"))
from fixture_matrix import MATRIX, fixture_path

import shapely.geometry as sg
from proto3.schema.input import BuildingInput
from proto3.schema.serialize import from_json


def load_fixture_polygon_mm(matrix_id):
    """Load fixture and return its first floor's footprint as a shapely Polygon (mm)."""
    b = from_json(BuildingInput, fixture_path(matrix_id))
    return sg.Polygon(b.floors[0].footprint)


for mid, meta in MATRIX.items():
    print(f"{mid:>3}  {meta['file']:<32}  expected_failure={meta.get('expected_failure')}")

In [ ]:
import numpy as np
from proto3.geometry.decompose import run


results = {}
for mid in MATRIX:
    poly = load_fixture_polygon_mm(mid)
    raw = run(poly)
    families = sorted({p['family_id'] for p in raw['pieces']})
    total_atom_area = sum(c.area for c, _ in raw['cells'])
    gap_pct = (poly.area - total_atom_area) / poly.area * 100
    results[mid] = {
        'polygon': poly,
        'raw': raw,
        'families': families,
        'gap_pct': gap_pct,
    }
    print(
        f"{mid}  pieces={len(raw['pieces'])}, families={len(families)}, "
        f"atoms={len(raw['cells'])}, gap={gap_pct:.4f}%"
    )

In [ ]:
from datetime import datetime

import matplotlib.pyplot as plt
from matplotlib.patches import Polygon as MplPolygon

FAMILY_COLORS = ['#9ad0c2', '#fdb462', '#a481c4', '#f4a3a3', '#88c4dc', '#c2d57a']


def plot_decomposition(ax, footprint, raw, title):
    pieces = raw['pieces']
    for cell, piece_id in raw['cells']:
        if cell.is_empty:
            continue
        fid = pieces[piece_id]['family_id']
        color = FAMILY_COLORS[fid % len(FAMILY_COLORS)]
        coords = list(cell.exterior.coords)
        ax.add_patch(MplPolygon(coords, facecolor=color, edgecolor='#444', linewidth=0.2, alpha=0.9))

    fx, fy = footprint.exterior.xy
    ax.plot(fx, fy, color='black', linewidth=1.8, zorder=10)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)
    families = sorted({p['family_id'] for p in pieces})
    n_atoms = sum(p['n_cells'] for p in pieces)
    ax.set_title(
        f"{title}\n#families={len(families)}, #pieces={len(pieces)}, #atoms={n_atoms}",
        fontsize=10,
    )


run_id = datetime.now().strftime('%Y%m%dT%H%M%S')
out_dir = ROOT / 'outputs' / 'notebooks' / 'step05_decomposition' / run_id
out_dir.mkdir(parents=True, exist_ok=True)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, mid in zip(axes.flat, MATRIX.keys()):
    res = results[mid]
    plot_decomposition(ax, res['polygon'], res['raw'], f"{mid} — {MATRIX[mid]['file']}")

fig.suptitle('Step 05 — v3.2 decomposition across 6 fixtures (mm-unit)', fontsize=13, fontweight='bold')
plt.tight_layout()
out_path = out_dir / 'step05_6_fixtures.png'
plt.savefig(out_path, dpi=110, bbox_inches='tight')
print('saved:', out_path.relative_to(ROOT))
plt.show()

---

## Notes

- All fixtures and outputs in this notebook are in **mm** (proto3 schema convention). `decompose.run()` wraps `auto_partition()` (m-unit v3.2 origin) with on-entry/on-exit conversion.
- **A1, R1**: identical 8000×6000mm footprint (R1 differs only in `program_request` — bathroom missing). Decomposition geometry is identical; the difference manifests at Stage 01 (cardinality fail).
- **B1**: L-shape — single family expected (axis-aligned throughout); two pieces (LIR + leftover), seamless cell phase via the per-family chain.
- **A2**: largest footprint (13000×10000mm); single family.
- **R2**: 4000×4000mm, used for Stage 02 area-gate fail (Step 06); decomposition itself succeeds.
- **D1**: ~20° rotated rectangle. Single family, theta auto-detected ≈ 20°. Validates v3.2 LIR rotation handling — atoms align with rotated footprint, boundary atoms are partial polygons clipped by the rotated edges.
- All six fixtures should report **gap < 1%** (effectively 0). Higher gap signals algorithm regression.
- Re-running creates a new `run_id` folder under `outputs/notebooks/step05_decomposition/`.
- Stage 00 normalization layer (Step 07 Plan §5 Def-14) will absorb broader unit conversion responsibilities (BuildingInput → run dispatch, ContactGraph mm-aware door checks).